# 05 — Stress and sensitivity

**Project:** *Selling Property Rental Ownership — A Hybrid Real Estate Model*  
**Author:** Dan Allouche · **Date:** May 2026

Stress overlays defined in the YAML config files (`stress_gfc.yaml`, `stress_covid.yaml`, `stress_rates2022.yaml`) are applied to the calibrated baseline to test the tranche structure under historically realistic crisis scenarios. We then run a parametric sensitivity to identify which inputs most strongly drive tranche fair value.


In [ ]:
from __future__ import annotations

import sys
from dataclasses import replace
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd

from tranche_pricing.config import load_config
from tranche_pricing.pricing import compare_credit_models, runner as pricing_runner
from tranche_pricing.viz.style import apply_style

apply_style()
pd.options.display.float_format = "{:,.4f}".format

baseline_cfg = load_config(ROOT / "config/paris_intermediate.yaml")
sim_baseline, gbm, vas, lgd, tranches, coupons = pricing_runner.build_inputs_from_yaml(baseline_cfg)


## Three stress overlays vs the baseline

For each YAML overlay we adjust the **price drift**, **default rate** and **short rate** as specified, then rerun the joint comparison. The table below summarises the senior, mezzanine and equity fair-to-par ratios under each scenario.

In [ ]:
from tranche_pricing.markets.price_gbm import GBMParams
from tranche_pricing.markets.rates_vasicek import VasicekParams

stress_files = ["stress_gfc.yaml", "stress_covid.yaml", "stress_rates2022.yaml"]
rows = []
for name in ["baseline", *[f.replace('.yaml', '') for f in stress_files]]:
    if name == "baseline":
        sim = sim_baseline; cur_gbm, cur_vas = gbm, vas
    else:
        stress_cfg = load_config(ROOT / f"config/{name}.yaml")
        overlay = stress_cfg.overlay
        cur_gbm = GBMParams(mu=gbm.mu + (overlay.price_dynamics.mu_shift_pct if overlay and overlay.price_dynamics else 0.0),
                            sigma=gbm.sigma * (overlay.price_dynamics.sigma_multiplier if overlay and overlay.price_dynamics else 1.0))
        cur_vas = VasicekParams(a=vas.a,
                                b=vas.b + (overlay.rates.delta_bps / 10000 if overlay and overlay.rates else 0.0),
                                sigma_r=vas.sigma_r)
        pd_mult = overlay.credit.pd_multiplier if overlay and overlay.credit else 1.0
        pd_t = min(0.95, 1 - (1 - baseline_cfg.credit.pd_annual_init * pd_mult) ** sim_baseline.horizon_years)
        sim = replace(sim_baseline, pd_terminal=pd_t)
    df, _ = compare_credit_models(sim, gbm=cur_gbm, vasicek=cur_vas, lgd=lgd, tranches=tranches, coupons=coupons, models=("gaussian_copula",))
    for _, r in df.iterrows():
        rows.append({"scenario": name, "instrument": r["instrument"], "fair_to_par": r["fair_to_par"], "mean_ann_return": r["mean_ann_return"], "risk_var_95": r["risk_var_95"]})

stress_df = pd.DataFrame(rows)
display(stress_df.pivot(index="instrument", columns="scenario", values="fair_to_par"))
display(stress_df.pivot(index="instrument", columns="scenario", values="mean_ann_return"))


## Parametric one-at-a-time sensitivity

For each key input we shift it by ±20 % around the baseline and record the impact on the senior, mezzanine and equity fair_to_par.

In [ ]:
sensitivity_rows = []
params = {
    "rho":  (baseline_cfg.credit.gaussian_copula.rho_init, lambda v: replace(sim_baseline, rho=v)),
    "pd_terminal": (sim_baseline.pd_terminal, lambda v: replace(sim_baseline, pd_terminal=v)),
    "gross_yield": (sim_baseline.gross_yield, lambda v: replace(sim_baseline, gross_yield=v)),
    "maintenance_pct": (sim_baseline.maintenance_pct, lambda v: replace(sim_baseline, maintenance_pct=v)),
}
for pname, (base_val, make) in params.items():
    for label, mult in [("down20", 0.8), ("baseline", 1.0), ("up20", 1.2)]:
        new_val = base_val * mult
        sim_pert = make(new_val)
        df, _ = compare_credit_models(sim_pert, gbm=gbm, vasicek=vas, lgd=lgd, tranches=tranches, coupons=coupons, models=("gaussian_copula",))
        for _, r in df[df["instrument"].isin(["equity", "mezzanine", "senior"])].iterrows():
            sensitivity_rows.append({"param": pname, "shift": label, "value": new_val, "instrument": r["instrument"], "fair_to_par": r["fair_to_par"]})

sens_df = pd.DataFrame(sensitivity_rows)
for inst in ["senior", "mezzanine", "equity"]:
    print(f"\n--- {inst.upper()} fair_to_par ---")
    print(sens_df[sens_df["instrument"] == inst].pivot(index="param", columns="shift", values="fair_to_par").to_string())


---
**Next:** the working paper (`report/main.tex`) consolidates the calibrated parameters, the comparison tables and these stress / sensitivity results into a self-contained reading.